# Bonus

🎯 84 feature’dan oluşan tam `ML_Houses_dataset.csv` dataset’iyle [buradan ulaşarak](https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv) serbestçe çalışabilirsiniz!

- Feature’ları inceleyin
- Uygun şekilde preprocess edin ve encode edin
- Feature engineering için beyin fırtınası yapın
- Bunları modelinize ekleyin
- Feature selection uygulayın

👇 Dosyayı yerel olarak `data` klasörüne kaydedin ve buradan içe aktarın.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

path = Path("data/ML_Houses_dataset.csv")
if not path.exists():
    url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv"
    pd.read_csv(url).to_csv(path, index=False)

df = pd.read_csv(path)
df.shape

(1760, 85)

## 1. Explore the data

In [12]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [13]:
# dtype split + missing values
print(df.dtypes.value_counts())

missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

object     45
int64      35
float64     5
Name: count, dtype: int64


WallMat         1755
PoolQC          1751
MiscFeature     1694
Alley           1648
Fence           1418
MasVnrType      1062
FireplaceQu      827
LotFrontage      309
GarageCond        93
GarageYrBlt       93
GarageType        93
GarageQual        93
GarageFinish      93
BsmtFinType2      45
BsmtExposure      44
BsmtFinType1      43
BsmtCond          43
BsmtQual          43
Pesos             13
RoofSurface       10
MasVnrArea         9
Electrical         1
dtype: int64

## 2. Drop the leaked target

In [14]:
# 'Pesos' == SalePrice * 20  → a leaked target (corr = 1.0). 'Id' is only a row identifier.
print("Pesos/SalePrice correlation:", df['Pesos'].corr(df['SalePrice']))

df = df.drop(columns=['Id', 'Pesos'])
df.shape

Pesos/SalePrice correlation: 1.0


(1760, 83)

## 3. Feature engineering

In [15]:
# A few domain-driven engineered features
df['TotalSF']   = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']  # total living area
df['HouseAge']  = df['YrSold'] - df['YearBuilt']                       # age of house at sale
df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath']               # weighted bathroom count

df[['TotalSF', 'HouseAge', 'TotalBath']].describe()

,TotalSF,HouseAge,TotalBath
count,1760.000000,1760.000000,1760.000000
mean,2587.172159,36.497727,1.756818
std,868.076300,30.130340,0.640192
min,334.000000,0.000000,0.000000
25%,2013.500000,8.000000,1.000000
50%,2475.000000,34.500000,2.000000
75%,3034.250000,55.000000,2.500000
max,11752.000000,136.000000,3.500000


## 4. Preprocess & baseline models

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

y = df['SalePrice']
X = df.drop(columns='SalePrice')

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='None')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), cat_cols),
])

for name, mdl in [('Ridge', Ridge()),
                  ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=1, n_jobs=-1))]:
    pipe = Pipeline([('pre', preprocessor), ('model', mdl)])
    r2 = cross_val_score(pipe, X, y, cv=5, scoring='r2').mean()
    print(f"{name}: R² = {r2:.4f}")

/home/samet/.pyenv/versions/workintech/lib/python3.12/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


Ridge: R² = 0.8803


/home/samet/.pyenv/versions/workintech/lib/python3.12/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


RandomForest: R² = 0.9226


## 5. Feature selection

In [17]:
# Rank features by RandomForest importance
pipe = Pipeline([('pre', preprocessor),
                 ('model', RandomForestRegressor(n_estimators=200, random_state=1, n_jobs=-1))])
pipe.fit(X, y)

feat_names = pipe.named_steps['pre'].get_feature_names_out()
importances = pd.Series(pipe.named_steps['model'].feature_importances_, index=feat_names)
importances.sort_values(ascending=False).head(20)

num__TotalSF         0.396209
num__OverallQual     0.363339
num__2ndFlrSF        0.042000
num__LotArea         0.012298
num__BsmtFinSF1      0.012010
num__YearBuilt       0.011545
num__HouseAge        0.011379
num__GarageCars      0.009052
num__GrLivArea       0.008363
num__GarageArea      0.007526
num__YearRemodAdd    0.006801
cat__BsmtQual_Ex     0.006599
num__BsmtUnfSF       0.006375
num__LotFrontage     0.006307
num__OverallCond     0.005438
num__MasVnrArea      0.004915
num__TotalBsmtSF     0.004906
num__TotalBath       0.004783
num__GarageYrBlt     0.003786
num__1stFlrSF        0.003764
dtype: float64

In [18]:
from sklearn.feature_selection import SelectFromModel

# Keep only the more important half of the (transformed) features, then re-evaluate
selected_pipe = Pipeline([
    ('pre', preprocessor),
    ('select', SelectFromModel(
        RandomForestRegressor(n_estimators=200, random_state=1, n_jobs=-1),
        threshold='median')),
    ('model', RandomForestRegressor(n_estimators=200, random_state=1, n_jobs=-1)),
])

r2_selected = cross_val_score(selected_pipe, X, y, cv=5, scoring='r2').mean()
print(f"After feature selection: R² = {r2_selected:.4f}")

/home/samet/.pyenv/versions/workintech/lib/python3.12/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


After feature selection: R² = 0.9231


ℹ️ Dataset’in açıklamasına mutlaka [buradan](https://drive.google.com/file/d/1qLxeQXufW_-KHOckpUweLPhitzjnP7H3/view?usp=sharing) referans verin.